# Handwriting Notes Digitizer: Colab/Kaggle launch

This notebook is a thin launcher for a small, reproducible compact image-to-LaTeX pilot. It does not download private or licensed data automatically. Put a verified JSONL manifest and its crop files on the runtime, and copy checkpoints to durable storage before the session ends.

In [ ]:
# Colab/Kaggle setup: adjust REPO_URL and DATA_DIR for your private dataset.
REPO_URL = 'https://github.com/your-org/Handwritten-Notes-Digitizer.git'
REPO_DIR = '/content/Handwritten-Notes-Digitizer'
# !git clone $REPO_URL $REPO_DIR
# %cd $REPO_DIR
# !pip install -r training/requirements-cuda.txt --index-url https://download.pytorch.org/whl/cu124

In [ ]:
# Point this at a verified MathWriting/HME/CROHME or personal-note manifest.
from pathlib import Path
import os
REPO_DIR = Path(os.environ.get('HND_REPO_DIR', '/content/Handwritten-Notes-Digitizer'))
MANIFEST = Path(os.environ.get('HND_FORMULA_MANIFEST', REPO_DIR / 'data/mathwriting_subset.jsonl'))
CHECKPOINT = Path(os.environ.get('HND_CHECKPOINT', '/content/checkpoints/compact_math.pt'))
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
assert MANIFEST.is_file(), f'Upload a verified manifest first: {MANIFEST}'

In [ ]:
# Run a short E4/E5 pilot. --epochs is a total epoch count when resuming.
import os, subprocess, sys
# The API package lives under backend/; keep the launcher portable when
# the notebook is executed from the repository root.
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'backend') + os.pathsep + env.get('PYTHONPATH', '')
command = [sys.executable, '-m', 'training.cli', '--manifest', str(MANIFEST), '--output', str(CHECKPOINT), '--epochs', '1', '--batch-size', '2', '--accumulate', '8', '--device', 'cuda']
# Add '--coverage' for E5, as a separate matched-budget run.
subprocess.run(command, cwd=REPO_DIR, env=env, check=True)

In [ ]:
# Record the checkpoint metadata and save it to durable storage.
import torch, json
payload = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
print(json.dumps(payload.get('metadata', {}), indent=2, sort_keys=True))
# Example durable copies:
# !cp $CHECKPOINT /content/drive/MyDrive/hnd-checkpoints/
# !cp $CHECKPOINT /kaggle/working/

## Evaluation reminder

Generate predictions on a verified development/test split and run `python -m benchmarks.runner`. Keep blank, failed, and truncated samples in the denominator. Do not call synthetic smoke data handwriting accuracy, and do not use the final test split to select a checkpoint.